# Understanding `saudi_patterns.py`, step by step

This notebook takes apart `cc_copilot/pii/saudi_patterns.py`, the file that finds and hides Saudi personal data
**before** any text is sent to Gemini.

Run the cells in order (Shift + Enter). Change the example texts as you go: that's the best way to learn.

**The big picture.** `mask_pii(text)` runs this pipeline:

```
text ──► 1. convert Arabic digits (١٢٣ → 123)
     ──► 2. regexes find candidates ("this looks like an ID")
     ──► 3. checksums reject look-alikes ("…but the check digit is wrong, so it's an order number")
     ──► 4. overlap rule ("this number is already part of an email, skip it")
     ──► 5. replace each hit with a placeholder such as [NATIONAL_ID]
```

| Part | Section |
|---|---|
| `_DIGIT_MAP`, `normalize_digits` | 1 |
| `PIIType`, `PIIMatch` | 2 |
| `is_valid_saudi_id` (Luhn) | 3 |
| `is_valid_iban` (mod-97) | 4 |
| `_PATTERNS` (the regexes) | 5 |
| `detect_pii` (puts it all together) | 6 |
| `mask_pii` | 7 |
| What it can't detect | 8 |
| Exercises | 9 |

## 0. Setup

In [1]:
import re
import sys
from pathlib import Path

# Make `cc_copilot` importable whether you start Jupyter from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from cc_copilot.pii.saudi_patterns import (
    _DIGIT_MAP,
    _PATTERNS,
    PIIMatch,
    PIIType,
    detect_pii,
    is_valid_iban,
    is_valid_saudi_id,
    mask_pii,
    normalize_digits,
)

print("Imported from:", ROOT / "cc_copilot" / "pii" / "saudi_patterns.py")

Imported from: /Users/xina/Downloads/contact-center-copilot/cc_copilot/pii/saudi_patterns.py


## 1. Arabic digits → normal digits

Saudi customers often type numbers with Arabic-Indic digits: `١٠٠٠٠٠٠٠٠٨` instead of `1000000008`.
To a computer these are **different characters**, so the regex `\d{10}`-style patterns written for `0-9` would miss them.

The fix: before searching, translate every Arabic digit to its normal digit.

In [2]:
str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

{1632: 48,
 1633: 49,
 1634: 50,
 1635: 51,
 1636: 52,
 1637: 53,
 1638: 54,
 1639: 55,
 1640: 56,
 1641: 57,
 1776: 48,
 1777: 49,
 1778: 50,
 1779: 51,
 1780: 52,
 1781: 53,
 1782: 54,
 1783: 55,
 1784: 56,
 1785: 57}

In [2]:
for ch in "٠١٢٣٤٥٦٧٨٩":
    print(f"{ch!r} is Unicode character U+{ord(ch):04X}  →  becomes {normalize_digits(ch)!r}")

'٠' is Unicode character U+0660  →  becomes '0'
'١' is Unicode character U+0661  →  becomes '1'
'٢' is Unicode character U+0662  →  becomes '2'
'٣' is Unicode character U+0663  →  becomes '3'
'٤' is Unicode character U+0664  →  becomes '4'
'٥' is Unicode character U+0665  →  becomes '5'
'٦' is Unicode character U+0666  →  becomes '6'
'٧' is Unicode character U+0667  →  becomes '7'
'٨' is Unicode character U+0668  →  becomes '8'
'٩' is Unicode character U+0669  →  becomes '9'


In [5]:
ex = "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹"
print(f"normalize_digits({ex!r}) = {normalize_digits(ex)!r}")
new=normalize_digits(ex)
print(new)

normalize_digits('٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹') = '01234567890123456789'
01234567890123456789


`str.maketrans(from, to)` builds a lookup table: the 1st character of `from` maps to the 1st of `to`, and so on.
The first 10 characters are the Arabic-Indic digits; the next 10 are the *Extended* (Persian/Urdu) digits, which some
keyboards produce. Both map to `0123456789`.

In [6]:
text = "رقم الهوية ١٠٠٠٠٠٠٠٠٨ وجوالي ۰۵۵۱۲۳۴۵۶۷"
normalized = normalize_digits(text)
print("original:  ", text)
print("normalized:", normalized)
print()
print("same length?", len(text) == len(normalized))

original:   رقم الهوية ١٠٠٠٠٠٠٠٠٨ وجوالي ۰۵۵۱۲۳۴۵۶۷
normalized: رقم الهوية 1000000008 وجوالي 0551234567

same length? True


**Why "same length" matters.** Each Arabic digit becomes exactly *one* normal digit, so position 11 in the normalized
text is position 11 in the original. That lets the code search the normalized text but cut the value out of the
**original** text, so the customer's message is returned exactly as they wrote it:

In [4]:
start = normalized.index("1000000008")
end = start + 10
print("found at positions", start, "to", end)
print("normalized[start:end] =", normalized[start:end])
print("original[start:end]   =", text[start:end], "  ← same position, original spelling")

found at positions 11 to 21
normalized[start:end] = 1000000008
original[start:end]   = ١٠٠٠٠٠٠٠٠٨   ← same position, original spelling


## 2. `PIIType` and `PIIMatch`: how a result is stored

`PIIType` is an **enum**: a fixed list of allowed labels. Using it instead of plain strings means a typo such as
`"NATIONL_ID"` is an error instead of a silent bug. `StrEnum` means each value also behaves like a normal string.

In [7]:
print(list(PIIType))
print(PIIType.IBAN == "IBAN")       # behaves like a string


[<PIIType.NATIONAL_ID: 'NATIONAL_ID'>, <PIIType.IQAMA: 'IQAMA'>, <PIIType.IBAN: 'IBAN'>, <PIIType.MOBILE: 'MOBILE'>, <PIIType.EMAIL: 'EMAIL'>]
True


In [8]:
print(f"[{PIIType.MOBILE}]")        # this is how the mask placeholder is built

[MOBILE]


In [11]:
PIIType.NATIONAL_ID

<PIIType.NATIONAL_ID: 'NATIONAL_ID'>

`PIIMatch` is a **dataclass**: a small container that records *what* was found and *where*.
`frozen=True` makes it read-only, so a match can't be changed by accident after it's created.

In [12]:
m = PIIMatch(type=PIIType.NATIONAL_ID, start=11, end=21, value="١٠٠٠٠٠٠٠٠٨")
print(m)



PIIMatch(type=<PIIType.NATIONAL_ID: 'NATIONAL_ID'>, start=11, end=21, value='١٠٠٠٠٠٠٠٠٨')


In [13]:
try:
    m.start = 0
except Exception as e:
    print("Changing it fails:", type(e).__name__, "-", e)

Changing it fails: FrozenInstanceError - cannot assign to field 'start'


## 3. The Saudi ID checksum (Luhn algorithm)

A national ID (starts with **1**) or Iqama (starts with **2**) has 10 digits. The **last digit is a check digit**,
calculated from the first nine. This lets us tell a real ID apart from any random 10-digit number, such as an
order number.

The rule (the Luhn algorithm, also used for bank cards):
1. Go through the digits. Positions 0, 2, 4, 6, 8 (the 1st, 3rd, 5th… digit) are **doubled**.
   If doubling gives a two-digit number, add its digits together (14 → 1 + 4 = 5).
2. Other positions are kept as they are.
3. Add everything up. The number is valid if the total ends in **0** (total % 10 == 0).

This helper does exactly what `is_valid_saudi_id` does, but prints every step:

In [7]:
def explain_saudi_id(number: str) -> None:
    print(f"Checking {number}\n")
    print(f"{'pos':>3}  {'digit':>5}  {'rule':<24} adds")
    total = 0
    for i, ch in enumerate(number):
        d = int(ch)
        if i % 2 == 0:
            doubled = d * 2
            adds = doubled // 10 + doubled % 10
            rule = f"double: {d}×2 = {doubled}" + (f" → {doubled // 10}+{doubled % 10}" if doubled > 9 else "")
        else:
            adds = d
            rule = "keep"
        total += adds
        print(f"{i:>3}  {d:>5}  {rule:<24} {adds:>3}")
    verdict = "VALID" if total % 10 == 0 else "INVALID"
    print(f"\ntotal = {total}, total % 10 = {total % 10}  →  {verdict}")
    print("is_valid_saudi_id says:", is_valid_saudi_id(number))


explain_saudi_id("1045238712")

Checking 1045238712

pos  digit  rule                     adds
  0      1  double: 1×2 = 2            2
  1      0  keep                       0
  2      4  double: 4×2 = 8            8
  3      5  keep                       5
  4      2  double: 2×2 = 4            4
  5      3  keep                       3
  6      8  double: 8×2 = 16 → 1+6     7
  7      7  keep                       7
  8      1  double: 1×2 = 2            2
  9      2  keep                       2

total = 40, total % 10 = 0  →  VALID
is_valid_saudi_id says: True


Now an **order number** that happens to be 10 digits and start with 1:

In [8]:
explain_saudi_id("1234567890")

Checking 1234567890

pos  digit  rule                     adds
  0      1  double: 1×2 = 2            2
  1      2  keep                       2
  2      3  double: 3×2 = 6            6
  3      4  keep                       4
  4      5  double: 5×2 = 10 → 1+0     1
  5      6  keep                       6
  6      7  double: 7×2 = 14 → 1+4     5
  7      8  keep                       8
  8      9  double: 9×2 = 18 → 1+8     9
  9      0  keep                       0

total = 43, total % 10 = 3  →  INVALID
is_valid_saudi_id says: False


The total doesn't end in 0, so it's rejected. Without this check, every order number, tracking number and
invoice number starting with 1 or 2 would be masked as an ID.

The function also rejects anything that isn't 10 digits or doesn't start with 1 or 2, *before* doing any maths:

In [9]:
for candidate in ["1045238712", "1045238713", "3045238712", "104523871", "10452387X2"]:
    print(f"{candidate:<12} → {is_valid_saudi_id(candidate)}")

1045238712   → True
1045238713   → False
3045238712   → False
104523871    → False
10452387X2   → False


**How were the fake IDs in the eval dataset made?** Take any 9 digits and try all 10 possible last digits.
Exactly one passes. This is how you can invent test IDs that pass the check but belong to nobody:

In [10]:
def make_test_id(first_nine: str) -> str:
    for last in "0123456789":
        if is_valid_saudi_id(first_nine + last):
            return first_nine + last


print(make_test_id("100000000"))   # → 1000000008, used in the tests
print(make_test_id("200000000"))   # → 2000000006, an Iqama
print(make_test_id("123456789"))   # try your own

1000000008
2000000006
1234567897


## 4. The IBAN checksum (mod-97)

A Saudi IBAN looks like `SA03 8000 0000 6080 1016 7519`: **24 characters**, starting with `SA`.
The two digits after `SA` (here `03`) are check digits. The international rule (ISO 13616):

1. Remove spaces.
2. Move the first 4 characters (`SA03`) to the end.
3. Replace letters with numbers: A = 10, B = 11, … S = 28, … Z = 35.
4. Treat the result as one huge number. The IBAN is valid if dividing by 97 leaves a remainder of **1**.

In [11]:
def explain_iban(iban: str) -> None:
    compact = re.sub(r"\s", "", iban).upper()
    print("1. remove spaces:        ", compact, f"({len(compact)} characters)")
    rearranged = compact[4:] + compact[:4]
    print("2. first 4 moved to end: ", rearranged)
    print("3. letters → numbers:     S =", int("S", 36), " A =", int("A", 36))
    numeric = "".join(str(int(ch, 36)) for ch in rearranged)
    print("   result:               ", numeric)
    remainder = int(numeric) % 97
    print("4. remainder after ÷ 97: ", remainder, "→", "VALID" if remainder == 1 else "INVALID")
    print("is_valid_iban says:", is_valid_iban(iban))


explain_iban("SA03 8000 0000 6080 1016 7519")

1. remove spaces:         SA0380000000608010167519 (24 characters)
2. first 4 moved to end:  80000000608010167519SA03
3. letters → numbers:     S = 28  A = 10
   result:                80000000608010167519281003
4. remainder after ÷ 97:  1 → VALID
is_valid_iban says: True


**The `int(ch, 36)` trick.** `int("7", 36)` reads a character as a digit in base 36, where `0-9` are 0-9 and
`A-Z` are 10-35. That's exactly the letter-to-number table the IBAN rule needs, in one line.

Change one digit and the check fails, which is how typos and random numbers get rejected:

In [12]:
explain_iban("SA04 8000 0000 6080 1016 7519")

1. remove spaces:         SA0480000000608010167519 (24 characters)
2. first 4 moved to end:  80000000608010167519SA04
3. letters → numbers:     S = 28  A = 10
   result:                80000000608010167519281004
4. remainder after ÷ 97:  2 → INVALID
is_valid_iban says: False


## 5. The regexes: finding candidates

`_PATTERNS` is a list of `(type, regex)` pairs. Regexes are only the **first filter**: they find text that *looks*
like PII. The checksums from sections 3 and 4 then decide.

In [14]:
_PATTERNS

[(<PIIType.EMAIL: 'EMAIL'>,
  re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', re.UNICODE)),
 (<PIIType.IBAN: 'IBAN'>,
  re.compile(r'\bSA\d{2}(?:\s?[0-9A-Z]{4}){5}\b', re.IGNORECASE|re.UNICODE)),
 (None, re.compile(r'(?<!\d)[12]\d{9}(?!\d)', re.UNICODE)),
 (<PIIType.MOBILE: 'MOBILE'>,
  re.compile(r'(?<![\d+])(?:(?:\+|00)966[\s-]?|0)?5\d(?:[\s-]?\d){7}(?!\d)',
             re.UNICODE))]

In [ ]:
for pii_type, pattern in _PATTERNS:
    print(f"{str(pii_type or 'ID / IQAMA'):<11} {pattern.pattern}")

EMAIL_RE, IBAN_RE, ID_RE, MOBILE_RE = (pattern for _, pattern in _PATTERNS)


EMAIL       [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}
IBAN        \bSA\d{2}(?:\s?[0-9A-Z]{4}){5}\b
ID / IQAMA  (?<!\d)[12]\d{9}(?!\d)
MOBILE      (?<![\d+])(?:(?:\+|00)966[\s-]?|0)?5\d(?:[\s-]?\d){7}(?!\d)


re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', re.UNICODE)

In [ ]:
def show_matches(pattern: re.Pattern, text: str) -> None:
    found = [m.group() for m in pattern.finditer(normalize_digits(text))]
    print(f"{text!r:<42} → {found}")

In [16]:
for m in ID_RE.finditer(normalize_digits("هويتي 1045238712")):
    print(m)

<re.Match object; span=(6, 16), match='1045238712'>


### 5a. National ID / Iqama: `(?<!\d)[12]\d{9}(?!\d)`

| Piece | Meaning |
|---|---|
| `(?<!\d)` | **lookbehind**: the character *before* must not be a digit |
| `[12]` | first digit is 1 (citizen) or 2 (resident) |
| `\d{9}` | then exactly 9 more digits |
| `(?!\d)` | **lookahead**: the character *after* must not be a digit |

The lookarounds stop it from grabbing 10 digits out of the middle of a longer number.
They check the neighbours **without including them** in the match.

In [14]:
show_matches(ID_RE, "هويتي 1045238712")
show_matches(ID_RE, "١٠٤٥٢٣٨٧١٢")                # Arabic digits, after normalising
show_matches(ID_RE, "card 41045238712000")        # inside a longer number → no match
show_matches(ID_RE, "3045238712")                 # starts with 3 → no match
show_matches(ID_RE, "order 1234567890")           # regex matches! the checksum rejects it later

'هويتي 1045238712'                         → ['1045238712']
'١٠٤٥٢٣٨٧١٢'                               → ['1045238712']
'card 41045238712000'                      → []
'3045238712'                               → []
'order 1234567890'                         → ['1234567890']


Note the last line: the regex **does** match the order number. That's expected, because regexes can't do maths.
`detect_pii` throws it out with the checksum (you'll see this in section 6).

### 5b. Mobile: `(?<![\d+])(?:(?:\+|00)966[\s-]?|0)?5\d(?:[\s-]?\d){7}(?!\d)`

| Piece | Meaning |
|---|---|
| `(?<![\d+])` | not preceded by a digit or `+` |
| `(?:\+\|00)966[\s-]?` | the country code `+966` or `00966`, optionally followed by a space or dash |
| `\|0` | …or the local prefix `0` |
| `( … )?` | the whole prefix is optional |
| `5\d` | Saudi mobiles start with 5, then one more digit (50, 53, 55, 56…) |
| `(?:[\s-]?\d){7}` | 7 more digits, each optionally preceded by a space or dash |
| `(?!\d)` | not followed by another digit |

`(?: … )` is a group that doesn't capture. It's only there so `?` or `{7}` can apply to the whole group.

In [15]:
for phone in ["0551234567", "+966551234567", "00966 55 123 4567", "+966-55-123-4567", "551234567"]:
    show_matches(MOBILE_RE, f"جوالي {phone}")

print()
show_matches(MOBILE_RE, "المكتب 0112345678")   # landline (starts 01, not 05) → no match
show_matches(MOBILE_RE, "0551234")              # too short → no match

'جوالي 0551234567'                         → ['0551234567']
'جوالي +966551234567'                      → ['+966551234567']
'جوالي 00966 55 123 4567'                  → ['00966 55 123 4567']
'جوالي +966-55-123-4567'                   → ['+966-55-123-4567']
'جوالي 551234567'                          → ['551234567']

'المكتب 0112345678'                        → []
'0551234'                                  → []


### 5c. IBAN: `\bSA\d{2}(?:\s?[0-9A-Z]{4}){5}\b` (case-insensitive)

`SA` + 2 check digits + five groups of 4 characters, each group optionally preceded by a space.
That's 2 + 2 + 20 = 24 characters. `\b` is a **word boundary**: the IBAN can't be glued to other letters or digits.

### 5d. Email: `[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}`

Some allowed characters, then `@`, then a domain, a dot, and at least 2 letters (`.com`, `.sa`).

In [16]:
show_matches(IBAN_RE, "IBAN: SA03 8000 0000 6080 1016 7519")
show_matches(IBAN_RE, "iban sa0380000000608010167519")      # lowercase works (IGNORECASE)
show_matches(EMAIL_RE, "راسلني على sara.k@example.com")

'IBAN: SA03 8000 0000 6080 1016 7519'      → ['SA03 8000 0000 6080 1016 7519']
'iban sa0380000000608010167519'            → ['sa0380000000608010167519']
'راسلني على sara.k@example.com'            → ['sara.k@example.com']


## 6. `detect_pii`: putting it together

For every pattern, in order, `detect_pii`:
1. finds every regex match in the **normalized** text,
2. **skips** it if it overlaps something already found,
3. **skips** it if the checksum fails (IBAN and ID only),
4. for IDs, decides `NATIONAL_ID` (starts with 1) or `IQAMA` (starts with 2),
5. records the match using the **original** text's spelling.

This tracer runs the same loop and prints every decision:

In [17]:
def trace_detect(text: str) -> None:
    normalized = normalize_digits(text)
    taken = []
    for pii_type, pattern in _PATTERNS:
        name = str(pii_type or "ID/IQAMA")
        for m in pattern.finditer(normalized):
            start, end = m.span()
            candidate = m.group()
            label = f"[{name:<8}] {candidate!r} at {start}-{end}"
            if any(start < t_end and t_start < end for t_start, t_end in taken):
                print(f"{label}: SKIP, overlaps something already found")
                continue
            if pii_type is PIIType.IBAN and not is_valid_iban(candidate):
                print(f"{label}: SKIP, IBAN checksum failed")
                continue
            if pii_type is None and not is_valid_saudi_id(candidate):
                print(f"{label}: SKIP, ID checksum failed")
                continue
            print(f"{label}: KEEP")
            taken.append((start, end))


trace_detect("طلب 1234567890، هويتي ١٠٤٥٢٣٨٧١٢، ايميلي 1000000008@example.com، جوالي 0551234567")

[EMAIL   ] '1000000008@example.com' at 41-63: KEEP
[ID/IQAMA] '1234567890' at 4-14: SKIP, ID checksum failed
[ID/IQAMA] '1045238712' at 22-32: KEEP
[ID/IQAMA] '1000000008' at 41-51: SKIP, overlaps something already found
[MOBILE  ] '0551234567' at 71-81: KEEP


Read the output line by line:
- The email is found first, because `EMAIL` is first in `_PATTERNS`.
- `1000000008` inside the email *looks* like a valid ID, but it overlaps the email, so it's skipped.
  **This is why the order of `_PATTERNS` matters.** Without the overlap rule, the result would be a mess like
  `[NATIONAL_ID]@example.com` inside `[EMAIL]`.
- `1234567890` matches the ID regex but fails the checksum.

**How the overlap test works.** Two ranges `[start, end)` and `[t_start, t_end)` overlap when
`start < t_end and t_start < end`:

In [18]:
def overlaps(a, b):
    return a[0] < b[1] and b[0] < a[1]


print(overlaps((0, 10), (5, 15)))    # partly overlapping  → True
print(overlaps((0, 10), (2, 8)))     # one inside the other → True
print(overlaps((0, 10), (10, 20)))   # touching but not overlapping → False

True
True
False


And the real function's result. Note that `value` keeps the customer's original Arabic digits:

In [19]:
for match in detect_pii("طلب 1234567890، هويتي ١٠٤٥٢٣٨٧١٢، ايميلي 1000000008@example.com، جوالي 0551234567"):
    print(match)

PIIMatch(type=<PIIType.NATIONAL_ID: 'NATIONAL_ID'>, start=22, end=32, value='١٠٤٥٢٣٨٧١٢')
PIIMatch(type=<PIIType.EMAIL: 'EMAIL'>, start=41, end=63, value='1000000008@example.com')
PIIMatch(type=<PIIType.MOBILE: 'MOBILE'>, start=71, end=81, value='0551234567')


## 7. `mask_pii`: replacing the values

`mask_pii` calls `detect_pii`, then replaces each match with `[TYPE]`. The important detail is that it works
**from the last match to the first** (`reversed(matches)`). Why? Watch what happens if you go forwards:

In [20]:
text = "هويتي 1045238712 وجوالي 0551234567"
matches = detect_pii(text)

wrong = text
for m in matches:  # forwards: WRONG
    wrong = wrong[: m.start] + f"[{m.type}]" + wrong[m.end :]
print("forwards: ", wrong)

masked, _ = mask_pii(text)  # the real function goes backwards
print("backwards:", masked)

forwards:  هويتي [NATIONAL_ID] وجوا[MOBILE]567
backwards: هويتي [NATIONAL_ID] وجوالي [MOBILE]


`[NATIONAL_ID]` is 13 characters, but the ID it replaces is 10. After the first replacement, everything after it
moves 3 places to the right, so the saved positions for the phone number are now wrong and the wrong text gets cut.
Going backwards, each replacement only shifts text that has **already been handled**, so the remaining positions
stay correct.

In [21]:
masked, matches = mask_pii(
    "Agent: تمام، الهوية ١٠٨٨٩٩٢٣١٦ والجوال 0501112233 والآيبان SA0380000000608010167519 وايميلك sara@example.com"
)
print(masked)
print()
for m in matches:
    print(f"{m.type:<12} {m.value}")

Agent: تمام، الهوية [NATIONAL_ID] والجوال [MOBILE] والآيبان [IBAN] وايميلك [EMAIL]

NATIONAL_ID  ١٠٨٨٩٩٢٣١٦
MOBILE       0501112233
IBAN         SA0380000000608010167519
EMAIL        sara@example.com


## 8. What it can't detect (know these for interviews)

Rules only catch what they're written for. These are **not** detected:

In [22]:
not_detected = [
    "هويتي 1045 238 712",                   # ID written with spaces
    "هويتي واحد صفر اربعة خمسة...",          # numbers spelled out in words
    "هاتف المكتب 0112345678",               # landlines (not covered on purpose)
    "اسمي سارة القحطاني",                    # names
    "ساكنة في حي النرجس، الرياض",            # addresses
]
for text in not_detected:
    print(f"{text!r:<45} → {detect_pii(text)}")

'هويتي 1045 238 712'                          → []
'هويتي واحد صفر اربعة خمسة...'                → []
'هاتف المكتب 0112345678'                      → []
'اسمي سارة القحطاني'                          → []
'ساكنة في حي النرجس، الرياض'                  → []


Names and addresses can't be found reliably with regexes, which is why the chat review agent *also* asks the LLM
to flag `contains_other_pii`. **Rules for structured data, AI for free text:** that combination is a good
design point to explain in an interview.

## 9. Exercises

1. **Your own text.** Change `my_text` below to a message in your dialect, with a mix of real-looking and fake
   numbers. Predict the output first, then run it.
2. **Break the checksum.** Use `make_test_id` to create an ID, change one digit, and check that `detect_pii`
   no longer finds it. Try changing two digits: can you find a pair of changes that still passes? (Luhn catches
   every single-digit typo, but not every double change.)
3. **Spaced IDs.** Section 8 shows `1045 238 712` isn't detected. Try writing a regex that allows optional
   spaces, like the mobile pattern does. What new false positives might it create?
4. **Add a test.** Pick one surprising result from this notebook and add it as a test in `tests/test_saudi_pii.py`
   or as a case in `evals/datasets/pii_cases.jsonl`.

In [23]:
my_text = "السلام عليكم، رقم إقامتي 2099812345 وجوالي +966 50 111 2233 ورقم الطلب 2000000001"

trace_detect(my_text)
print()
print(mask_pii(my_text)[0])

[ID/IQAMA] '2099812345' at 25-35: KEEP
[ID/IQAMA] '2000000001' at 71-81: SKIP, ID checksum failed
[MOBILE  ] '+966 50 111 2233' at 43-59: KEEP

السلام عليكم، رقم إقامتي [IQAMA] وجوالي [MOBILE] ورقم الطلب 2000000001
